# CourtVision — Phase 3: Court Detection & Spatial Calibration
**Kaggle / GPU optional for calibration; GPU recommended for tracking.**

Goal: convert player **footpoints** from video pixels into a schematic 2D basketball court using a verified homography. Includes line diagnostics, manual correspondence setup, validation, player tracking, overlay, top-down visualization, and export.

**Important:** SportsMOT sequences can be broadcast footage with camera motion, partial court visibility, and unknown court markings. A single fixed homography is valid only for a stationary camera and planar court. If camera moves, re-calibrate per shot or frame; do not report metric distances from an unverified transform. We do not infer corners from arbitrary image boundaries.


## 1 — Install and configure

In [ ]:
%pip -q install "ultralytics>=8.4" "opencv-python-headless>=4.9" "pandas>=2.2" "PyYAML>=6" "trackers"
%pip install -q ipympl

In [ ]:
from pathlib import Path
import json, shutil, zipfile, math, time
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt, yaml
from IPython.display import display
from ultralytics import YOLO

ROOT=Path('/kaggle/working/courtvision_phase3'); ROOT.mkdir(parents=True,exist_ok=True)
DATA_ROOT=Path('/kaggle/working/courtvision_data')
SEQUENCE_NAME='v_00HRwkvvjtQ_c007'
N_FRAMES=300
IMGSZ=1280; CONF=0.25; IOU=0.70; DEVICE=0
RUN_TRACKING=True  # False if you supply a compatible Phase 2C tracks_optimized.csv
EXISTING_TRACK_CSV=None  # e.g. '/kaggle/input/my-phase2c/tracks_optimized.csv'
COURT_LENGTH_M=28.0; COURT_WIDTH_M=15.0 # FIBA schematic; set to your actual court dimensions
FPS=25.0
print('Output:',ROOT)


## 2 — Locate footage and checkpoint

In [ ]:
def find_sequence():
    for root in [DATA_ROOT,Path('/kaggle/input')]:
        if not root.exists():continue
        for p in root.rglob(SEQUENCE_NAME):
            if (p/'img1').is_dir():return p
    return None
SEQ_DIR=find_sequence()
if SEQ_DIR is None:
    import subprocess
    DATA_ROOT.mkdir(parents=True,exist_ok=True)
    subprocess.run(['trackers','download','sportsmot','--split','val','--asset','frames,annotations','-o',str(DATA_ROOT)],check=True)
    SEQ_DIR=find_sequence()
assert SEQ_DIR is not None, 'SportsMOT video sequence not found'
FRAME_PATHS=sorted((SEQ_DIR/'img1').glob('*.jpg'))[:N_FRAMES]
assert FRAME_PATHS,'No frames found'
first=cv2.imread(str(FRAME_PATHS[0])); assert first is not None
H,W=first.shape[:2]
print('Sequence:',SEQ_DIR,'Frames:',len(FRAME_PATHS),'Resolution:',(W,H))
pts=sorted(Path('/kaggle/input').rglob('best.pt'))
BEST_PT=next((p for p in pts if 'courtvision' in str(p).lower()),pts[0] if pts else None)
if RUN_TRACKING:assert BEST_PT is not None,'Attach Kaggle input containing best.pt'
print('Checkpoint:',BEST_PT)


## 3 — Court line diagnostic (not a calibrated court detector)

In [ ]:
gray=cv2.cvtColor(first,cv2.COLOR_BGR2GRAY)
edges=cv2.Canny(gray,70,160)
segments=cv2.HoughLinesP(edges,1,np.pi/180,threshold=95,minLineLength=75,maxLineGap=20)
diagnostic=first.copy()
if segments is not None:
    for seg in segments[:,0]:
        x1,y1,x2,y2=map(int,seg); cv2.line(diagnostic,(x1,y1),(x2,y2),(0,255,255),2)
cv2.imwrite(str(ROOT/'court_line_diagnostic.jpg'),diagnostic)
plt.figure(figsize=(15,8));plt.imshow(cv2.cvtColor(diagnostic,cv2.COLOR_BGR2RGB));plt.title('Line candidates — may include stands, advertising, and player edges');plt.axis('off');plt.show()


## 4 — Set four **verified** image-to-court correspondences

**Action required:** Choose four **distinct, visible court-floor landmarks** whose real court coordinates you know (for example, line intersections). Record pixel coordinates `(x,y)` from the image and matching `(court_x_m,court_y_m)` in the top-down court system. Use a consistent FIBA-style rectangle `(0,0)` to `(28,15)` **only if the physical court actually matches these dimensions**. Avoid selecting corners that are offscreen, guessed, or occluded.

The cell below shows a coordinate grid for reading pixel positions. Fill in `IMAGE_POINTS` and `COURT_POINTS` **in corresponding order**. Leave placeholders unchanged to skip calibration safely.


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

frame = cv2.imread(str(FRAME_PATHS[0]))
rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(18, 10))
ax.imshow(rgb)

ax.set_xlim(0, 1280)
ax.set_ylim(720, 0)

ax.set_xticks(np.arange(0, 1281, 40))
ax.set_yticks(np.arange(0, 721, 40))

ax.grid(color="cyan", alpha=0.35, linewidth=0.6)
ax.set_title("CourtVision | Court Landmark Selection")

plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt

indices = [0, 75, 150, 225, 299]

fig, axes = plt.subplots(5, 1, figsize=(15, 35))

for ax, idx in zip(axes, indices):
    frame = cv2.imread(str(FRAME_PATHS[idx]))
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt

indices = [0, 75, 150, 225, 299]

fig, axes = plt.subplots(5, 1, figsize=(16, 35))

for ax, idx in zip(axes, indices):
    frame = cv2.imread(str(FRAME_PATHS[idx]))
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {idx}", fontsize=16)
    ax.axis("off")

plt.tight_layout()

# Save the comparison image
output = "/kaggle/working/courtvision_5_frames.png"
plt.savefig(output, dpi=150, bbox_inches="tight")
plt.show()

print("Saved:", output)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Select calibration frame
CALIBRATION_INDEX = 75

frame_path = FRAME_PATHS[CALIBRATION_INDEX]
frame = cv2.imread(str(frame_path))

assert frame is not None, "Could not load calibration frame"

# Save original resolution
output = Path("/kaggle/working/courtvision_calibration_frame75.png")

cv2.imwrite(str(output), frame)

print("Frame:", frame_path.name)
print("Resolution:", frame.shape[1], "x", frame.shape[0])
print("Saved:", output)

# Display
plt.figure(figsize=(18, 10))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title("CourtVision — Calibration Frame 75")
plt.axis("off")
plt.show()

In [ ]:
import cv2
import base64
import json
from IPython.display import display, HTML

frame = cv2.imread(str(FRAME_PATHS[75]))
assert frame is not None

success, encoded = cv2.imencode(".png", frame)
assert success

image_base64 = base64.b64encode(encoded).decode("utf-8")

html = f"""
<div style="font-family:Arial;max-width:1300px">
    <h3>CourtVision — Court Landmark Selector</h3>

    <p>Click the four court landmarks in this order:</p>
    <ol>
        <li>P1 — Baseline / lower lane boundary</li>
        <li>P2 — Baseline / upper lane boundary</li>
        <li>P3 — Free-throw line / upper lane boundary</li>
        <li>P4 — Free-throw line / lower lane boundary</li>
    </ol>

    <div style="position:relative;display:inline-block">
        <img id="court-image"
             src="data:image/png;base64,{image_base64}"
             style="width:100%;max-width:1280px;cursor:crosshair">
        <canvas id="court-overlay"
                width="{frame.shape[1]}"
                height="{frame.shape[0]}"
                style="position:absolute;left:0;top:0;width:100%;height:100%;pointer-events:none">
        </canvas>
    </div>

    <p><button onclick="resetCourtPoints()">Reset Points</button></p>

    <pre id="court-coordinates"
         style="background:#111;color:#00ff88;padding:15px">
Click on the court image to begin.
    </pre>
</div>

<script>
(function() {{
    const img = document.getElementById("court-image");
    const canvas = document.getElementById("court-overlay");
    const ctx = canvas.getContext("2d");
    const output = document.getElementById("court-coordinates");

    let points = [];

    img.addEventListener("click", function(event) {{
        if (points.length >= 4) return;

        const rect = img.getBoundingClientRect();

        const x = (event.clientX - rect.left) *
                  (img.naturalWidth / rect.width);

        const y = (event.clientY - rect.top) *
                  (img.naturalHeight / rect.height);

        points.push([Math.round(x), Math.round(y)]);

        drawPoints();
    }});

    function drawPoints() {{
        ctx.clearRect(0, 0, canvas.width, canvas.height);

        points.forEach((point, index) => {{
            ctx.beginPath();
            ctx.arc(point[0], point[1], 7, 0, 2 * Math.PI);
            ctx.fillStyle = "red";
            ctx.fill();
            ctx.strokeStyle = "white";
            ctx.lineWidth = 2;
            ctx.stroke();

            ctx.fillStyle = "yellow";
            ctx.font = "bold 20px Arial";
            ctx.fillText("P" + (index + 1),
                         point[0] + 12,
                         point[1] - 12);
        }});

        output.textContent =
            "Selected points:\\n" +
            points.map((p, i) =>
                "P" + (i + 1) + ": (" + p[0] + ", " + p[1] + ")"
            ).join("\\n") +
            "\\n\\nPython coordinates:\\n" +
            "IMAGE_POINTS = " + JSON.stringify(points);
    }}

    window.resetCourtPoints = function() {{
        points = [];
        drawPoints();
    }};
}})();
</script>
"""

display(HTML(html))

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Your eight selected points
LINE_POINTS = np.array([
    [261, 455],
    [287, 427],
    [356, 350],
    [486, 362],
    [375, 467],
    [683, 499],
    [789, 393],  # Upper free-throw corner
    [728, 500],  # Lower free-throw corner
], dtype=np.float64)

def make_line(a, b):
    return np.cross(
        [a[0], a[1], 1.0],
        [b[0], b[1], 1.0]
    )

def intersect(a, b):
    p = np.cross(a, b)

    if abs(p[2]) < 1e-8:
        raise ValueError("Lines are parallel.")

    return p[:2] / p[2]

# Identify court lines
baseline = make_line(LINE_POINTS[0], LINE_POINTS[1])
upper_lane = make_line(LINE_POINTS[2], LINE_POINTS[3])
lower_lane = make_line(LINE_POINTS[4], LINE_POINTS[5])

# Recover baseline intersections
baseline_upper = intersect(baseline, upper_lane)
baseline_lower = intersect(baseline, lower_lane)

# Construct four court corners
IMAGE_POINTS = np.array([
    baseline_lower,
    baseline_upper,
    LINE_POINTS[6],
    LINE_POINTS[7]
], dtype=np.float32)

print("Recovered image landmarks:")
for i, point in enumerate(IMAGE_POINTS, 1):
    print(f"P{i}: {point.round(2)}")

# Load frame
frame = cv2.imread(str(FRAME_PATHS[100]))
assert frame is not None

overlay = frame.copy()

# Draw the lane polygon
cv2.polylines(
    overlay,
    [np.round(IMAGE_POINTS).astype(np.int32)],
    isClosed=True,
    color=(0, 255, 0),
    thickness=3
)

# Draw corners
for i, (x, y) in enumerate(IMAGE_POINTS, 1):
    x, y = int(round(x)), int(round(y))

    cv2.circle(
        overlay, (x, y), 7,
        (0, 0, 255), -1
    )

    cv2.putText(
        overlay,
        f"P{i}",
        (x + 12, y - 12),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

# Display
plt.figure(figsize=(18, 10))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.title("CourtVision — Frame 100 Reconstructed Lane")
plt.axis("off")
plt.show()

# Save validation image
cv2.imwrite(
    "/kaggle/working/courtvision_frame100_validation.png",
    overlay
)

In [ ]:
import numpy as np

# Your manually selected landmarks from Frame 75
IMAGE_POINTS = np.array([
    [63, 468],   # P1: baseline / lower lane
    [174, 341],  # P2: baseline / upper lane
    [682, 386],  # P3: free-throw / upper lane
    [612, 508],  # P4: free-throw / lower lane
], dtype=np.float32)

# Court coordinates in metres, relative to the lane.
# Assumes a 5.79 m (19 ft) lane length and 3.66 m (12 ft) lane width.
# These dimensions MUST be verified for the actual court.
LANE_LENGTH = 5.79
LANE_WIDTH = 3.66

COURT_POINTS = np.array([
    [0, LANE_WIDTH],            # P1
    [0, 0],                     # P2
    [LANE_LENGTH, 0],           # P3
    [LANE_LENGTH, LANE_WIDTH],  # P4
], dtype=np.float32)

print("IMAGE_POINTS:", IMAGE_POINTS.shape)
print("COURT_POINTS:", COURT_POINTS.shape)
print("Landmark variables initialized.")

In [ ]:
import cv2
import numpy as np
import json
from pathlib import Path

# NCAA court dimensions
COURT_LENGTH_M = 28.6512
COURT_WIDTH_M = 15.24

# NCAA lane dimensions
LANE_LENGTH_M = 5.7912   # 19 feet
LANE_WIDTH_M = 3.6576    # 12 feet

# Physical coordinates relative to the left baseline
COURT_POINTS = np.array([
    [0, LANE_WIDTH_M],               # P1: baseline lower
    [0, 0],                          # P2: baseline upper
    [LANE_LENGTH_M, 0],              # P3: free-throw upper
    [LANE_LENGTH_M, LANE_WIDTH_M],   # P4: free-throw lower
], dtype=np.float32)

# Calculate homography
HOMOGRAPHY = cv2.getPerspectiveTransform(
    IMAGE_POINTS.astype(np.float32),
    COURT_POINTS
)

CALIBRATED = True
CALIBRATION_FRAME = 100

print("Calibration successful!")
print("\nHomography matrix:")
print(HOMOGRAPHY)

# Save
output_path = Path("/kaggle/working/courtvision_homography_frame100.json")

output_path.write_text(json.dumps({
    "frame": CALIBRATION_FRAME,
    "image_points": IMAGE_POINTS.tolist(),
    "court_points": COURT_POINTS.tolist(),
    "homography": HOMOGRAPHY.tolist(),
    "lane_length_m": LANE_LENGTH_M,
    "lane_width_m": LANE_WIDTH_M
}, indent=2))

print("\nSaved:", output_path)

In [ ]:
def image_to_court(x, y, H):
    point = np.array([[[x, y]]], dtype=np.float32)

    result = cv2.perspectiveTransform(point, H)

    return result[0, 0]


def player_bbox_to_court(bbox, H):
    """
    bbox format: [x1, y1, x2, y2]
    """
    x1, y1, x2, y2 = bbox

    foot_x = (x1 + x2) / 2
    foot_y = y2

    return image_to_court(foot_x, foot_y, H)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Reference and next frame
frame_a = cv2.imread(str(FRAME_PATHS[100]))
frame_b = cv2.imread(str(FRAME_PATHS[101]))

assert frame_a is not None and frame_b is not None

# Convert to grayscale
gray_a = cv2.cvtColor(frame_a, cv2.COLOR_BGR2GRAY)
gray_b = cv2.cvtColor(frame_b, cv2.COLOR_BGR2GRAY)

# Detect ORB features
orb = cv2.ORB_create(nfeatures=3000)

kp_a, des_a = orb.detectAndCompute(gray_a, None)
kp_b, des_b = orb.detectAndCompute(gray_b, None)

assert des_a is not None and des_b is not None, \
    "Insufficient visual features detected."

# Match features
matcher = cv2.BFMatcher(cv2.NORM_HAMMING)

matches = matcher.knnMatch(des_a, des_b, k=2)

# Lowe ratio test
good_matches = [
    m for m, n in matches
    if m.distance < 0.75 * n.distance
]

print("Good feature matches:", len(good_matches))

assert len(good_matches) >= 20, \
    "Not enough feature matches to estimate camera motion."

# Extract corresponding image points
pts_a = np.float32([
    kp_a[m.queryIdx].pt for m in good_matches
])

pts_b = np.float32([
    kp_b[m.trainIdx].pt for m in good_matches
])

# Estimate transformation: Frame 100 → Frame 101
motion_H, inlier_mask = cv2.findHomography(
    pts_a,
    pts_b,
    cv2.RANSAC,
    3.0
)

assert motion_H is not None, \
    "Camera motion estimation failed."

inliers = int(inlier_mask.sum())
inlier_ratio = inliers / len(good_matches)

print("RANSAC inliers:", inliers)
print("Inlier ratio:", round(inlier_ratio, 3))

# Transfer reference calibration to Frame 101
H_frame101 = HOMOGRAPHY @ np.linalg.inv(motion_H)

print("\nFrame 101 homography:")
print(H_frame101)

# Visualize feature matches
matched_image = cv2.drawMatches(
    frame_a,
    kp_a,
    frame_b,
    kp_b,
    good_matches[:50],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(18, 8))
plt.imshow(cv2.cvtColor(matched_image, cv2.COLOR_BGR2RGB))
plt.title("CourtVision — Feature Matching: Frame 100 → 101")
plt.axis("off")
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load Frame 101
frame101 = cv2.imread(str(FRAME_PATHS[101]))
assert frame101 is not None

# Transform the original Frame 100 lane corners into Frame 101
points_101 = cv2.perspectiveTransform(
    IMAGE_POINTS.reshape(-1, 1, 2).astype(np.float32),
    motion_H
).reshape(-1, 2)

# Draw projected lane
overlay = frame101.copy()

cv2.polylines(
    overlay,
    [np.round(points_101).astype(np.int32)],
    isClosed=True,
    color=(0, 255, 0),
    thickness=3
)

for i, (x, y) in enumerate(points_101, 1):
    x, y = int(round(x)), int(round(y))

    cv2.circle(overlay, (x, y), 7, (0, 0, 255), -1)

    cv2.putText(
        overlay,
        f"P{i}",
        (x + 10, y - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

# Compare Frame 100 and Frame 101
frame100 = cv2.imread(str(FRAME_PATHS[100]))

reference = frame100.copy()

cv2.polylines(
    reference,
    [np.round(IMAGE_POINTS).astype(np.int32)],
    True,
    (0, 255, 0),
    3
)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

axes[0].imshow(cv2.cvtColor(reference, cv2.COLOR_BGR2RGB))
axes[0].set_title("Frame 100 — Reference Calibration")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[1].set_title("Frame 101 — Transferred Calibration")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print("Frame 101 projected lane corners:")
print(np.round(points_101, 2))

In [ ]:
import cv2
import numpy as np
import json
from pathlib import Path
from tqdm.auto import tqdm

REFERENCE_FRAME = 100
N_FRAMES = min(300, len(FRAME_PATHS))

orb = cv2.ORB_create(nfeatures=3000)
matcher = cv2.BFMatcher(cv2.NORM_HAMMING)

# Image-to-court homographies
homographies = {REFERENCE_FRAME: HOMOGRAPHY.copy()}

# Court lane corners in reference frame
reference_corners = IMAGE_POINTS.astype(np.float32)

# Quality diagnostics
quality = {
    REFERENCE_FRAME: {
        "status": "reference",
        "matches": None,
        "inliers": None,
        "inlier_ratio": None
    }
}

def estimate_motion(idx_a, idx_b):
    """Estimate image motion from frame A to frame B."""

    img_a = cv2.imread(str(FRAME_PATHS[idx_a]))
    img_b = cv2.imread(str(FRAME_PATHS[idx_b]))

    if img_a is None or img_b is None:
        return None, {"status": "missing_frame"}

    gray_a = cv2.cvtColor(img_a, cv2.COLOR_BGR2GRAY)
    gray_b = cv2.cvtColor(img_b, cv2.COLOR_BGR2GRAY)

    kp_a, des_a = orb.detectAndCompute(gray_a, None)
    kp_b, des_b = orb.detectAndCompute(gray_b, None)

    if des_a is None or des_b is None:
        return None, {"status": "no_features"}

    raw_matches = matcher.knnMatch(des_a, des_b, k=2)

    good = [
        m for pair in raw_matches
        if len(pair) == 2
        for m, n in [pair]
        if m.distance < 0.75 * n.distance
    ]

    if len(good) < 30:
        return None, {
            "status": "insufficient_matches",
            "matches": len(good)
        }

    pts_a = np.float32([
        kp_a[m.queryIdx].pt for m in good
    ])

    pts_b = np.float32([
        kp_b[m.trainIdx].pt for m in good
    ])

    M, mask = cv2.findHomography(
        pts_a,
        pts_b,
        cv2.RANSAC,
        3.0
    )

    if M is None or mask is None:
        return None, {"status": "ransac_failed"}

    inliers = int(mask.sum())
    ratio = inliers / len(good)

    metrics = {
        "matches": len(good),
        "inliers": inliers,
        "inlier_ratio": round(ratio, 4)
    }

    if inliers < 25 or ratio < 0.55:
        metrics["status"] = "low_confidence"
        return None, metrics

    # Verify that the reference lane geometry remains reasonable
    # under this adjacent-frame transformation.
    h, w = img_a.shape[:2]

    test_points = np.float32([
        [w * 0.25, h * 0.25],
        [w * 0.75, h * 0.25],
        [w * 0.75, h * 0.75],
        [w * 0.25, h * 0.75]
    ]).reshape(-1, 1, 2)

    transformed = cv2.perspectiveTransform(
        test_points, M
    ).reshape(-1, 2)

    if not np.isfinite(transformed).all():
        metrics["status"] = "invalid_geometry"
        return None, metrics

    displacement = np.linalg.norm(
        transformed - test_points.reshape(-1, 2),
        axis=1
    )

    # Reject implausibly large adjacent-frame changes.
    if displacement.max() > 150:
        metrics["status"] = "excessive_motion"
        return None, metrics

    metrics["status"] = "ok"
    metrics["max_displacement"] = float(displacement.max())

    return M, metrics


def propagate(start, stop, step):
    """Propagate calibration from one frame to adjacent frames."""

    for idx in tqdm(
        range(start, stop, step),
        desc="Forward" if step > 0 else "Backward"
    ):
        previous = idx - step

        if previous not in homographies:
            quality[idx] = {"status": "uncalibrated"}
            continue

        M, metrics = estimate_motion(previous, idx)
        quality[idx] = metrics

        if M is None:
            continue

        try:
            M_inv = np.linalg.inv(M)

            # H_previous maps previous image -> court.
            # M_inv maps current image -> previous image.
            H_current = homographies[previous] @ M_inv

            if not np.isfinite(H_current).all():
                quality[idx]["status"] = "invalid_homography"
                continue

            if abs(H_current[2, 2]) < 1e-10:
                quality[idx]["status"] = "invalid_scale"
                continue

            H_current /= H_current[2, 2]

            homographies[idx] = H_current

        except np.linalg.LinAlgError:
            quality[idx]["status"] = "singular_motion"


# Forward: 100 -> 299
propagate(101, N_FRAMES, 1)

# Backward: 100 -> 0
propagate(99, -1, -1)

print("\nDynamic calibration completed")
print("Total frames:", N_FRAMES)
print("Calibrated frames:", len(homographies))
print("Uncalibrated frames:", N_FRAMES - len(homographies))

# Save results
output_path = Path(
    "/kaggle/working/courtvision_dynamic_homographies.json"
)

output_path.write_text(
    json.dumps({
        "reference_frame": REFERENCE_FRAME,
        "homographies": {
            str(k): v.tolist()
            for k, v in homographies.items()
        },
        "quality": {
            str(k): v
            for k, v in quality.items()
        }
    }, indent=2)
)

print("Saved:", output_path)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Frames to inspect
CHECK_FRAMES = [0, 50, 100, 150, 200, 250, 299]

fig, axes = plt.subplots(2, 4, figsize=(22, 11))
axes = axes.flatten()

# Physical coordinates of the free-throw lane
lane_court_points = COURT_POINTS.astype(np.float32)

for ax, frame_idx in zip(axes, CHECK_FRAMES):

    frame = cv2.imread(str(FRAME_PATHS[frame_idx]))
    H = homographies.get(frame_idx)

    if frame is None or H is None:
        ax.set_title(f"Frame {frame_idx} — unavailable")
        ax.axis("off")
        continue

    # Court coordinates -> image coordinates
    try:
        H_inv = np.linalg.inv(H)

        projected = cv2.perspectiveTransform(
            lane_court_points.reshape(-1, 1, 2),
            H_inv
        ).reshape(-1, 2)

        overlay = frame.copy()

        if np.isfinite(projected).all():
            cv2.polylines(
                overlay,
                [np.round(projected).astype(np.int32)],
                isClosed=True,
                color=(0, 255, 0),
                thickness=3
            )

            for i, (x, y) in enumerate(projected, 1):
                cv2.circle(
                    overlay,
                    (int(round(x)), int(round(y))),
                    5,
                    (0, 0, 255),
                    -1
                )

        ax.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Frame {frame_idx}")
        ax.axis("off")

    except np.linalg.LinAlgError:
        ax.set_title(f"Frame {frame_idx} — invalid H")
        ax.axis("off")

for ax in axes[len(CHECK_FRAMES):]:
    ax.axis("off")

plt.suptitle(
    "CourtVision — Dynamic Homography Validation",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

frame = cv2.imread(str(FRAME_PATHS[100]))
h, w = frame.shape[:2]

# Approximate visible wooden-court region for Frame 100.
# This is a starting mask, not a universal court segmentation.
polygon = np.array([
    [60, 665],
    [510, 145],
    [1279, 245],
    [1279, 719],
    [60, 719]
], dtype=np.int32)

mask = np.zeros((h, w), dtype=np.uint8)
cv2.fillPoly(mask, [polygon], 255)

masked = cv2.bitwise_and(frame, frame, mask=mask)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original Frame 100")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(masked, cv2.COLOR_BGR2RGB))
axes[1].set_title("Court Region Mask")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def create_court_mask(frame):
    """
    Estimate the wooden basketball court region using HSV color.
    Returns a binary mask.
    """
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Broad range for warm wooden-floor colors.
    lower = np.array([5, 25, 65], dtype=np.uint8)
    upper = np.array([35, 255, 255], dtype=np.uint8)

    mask = cv2.inRange(hsv, lower, upper)

    # Remove isolated noise and fill small gaps
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (9, 9)
    )

    mask = cv2.morphologyEx(
        mask, cv2.MORPH_CLOSE, kernel, iterations=2
    )

    mask = cv2.morphologyEx(
        mask, cv2.MORPH_OPEN, kernel, iterations=1
    )

    # Retain large connected regions only
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask, connectivity=8
    )

    cleaned = np.zeros_like(mask)

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]

        if area > 10000:
            cleaned[labels == i] = 255

    return cleaned


# Test across multiple camera positions
TEST_FRAMES = [0, 50, 100, 150, 200, 250, 299]

fig, axes = plt.subplots(
    len(TEST_FRAMES), 2,
    figsize=(15, 24)
)

for row, idx in enumerate(TEST_FRAMES):
    frame = cv2.imread(str(FRAME_PATHS[idx]))

    mask = create_court_mask(frame)

    masked = cv2.bitwise_and(frame, frame, mask=mask)

    axes[row, 0].imshow(
        cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    )
    axes[row, 0].set_title(f"Frame {idx} — Original")

    axes[row, 1].imshow(
        cv2.cvtColor(masked, cv2.COLOR_BGR2RGB)
    )
    axes[row, 1].set_title(f"Frame {idx} — Automatic Court Mask")

    for ax in axes[row]:
        ax.axis("off")
# Save the complete 7-frame court segmentation comparison
output_path = "/kaggle/working/courtvision_court_segmentation.png"

plt.tight_layout()
plt.savefig(
    output_path,
    dpi=200,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(f"Image saved to: {output_path}")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Frames to compare
idx_a = 100
idx_b = 101

frame_a = cv2.imread(str(FRAME_PATHS[idx_a]))
frame_b = cv2.imread(str(FRAME_PATHS[idx_b]))

assert frame_a is not None and frame_b is not None

# Generate court masks
mask_a = create_court_mask(frame_a)
mask_b = create_court_mask(frame_b)

# Exclude bottom broadcast graphics
h, w = frame_a.shape[:2]
mask_a[int(h * 0.85):, :] = 0
mask_b[int(h * 0.85):, :] = 0

# Detect court-only features
orb = cv2.ORB_create(nfeatures=3000)

gray_a = cv2.cvtColor(frame_a, cv2.COLOR_BGR2GRAY)
gray_b = cv2.cvtColor(frame_b, cv2.COLOR_BGR2GRAY)

kp_a, des_a = orb.detectAndCompute(gray_a, mask_a)
kp_b, des_b = orb.detectAndCompute(gray_b, mask_b)

assert des_a is not None and des_b is not None

# Match features
matcher = cv2.BFMatcher(cv2.NORM_HAMMING)

raw_matches = matcher.knnMatch(des_a, des_b, k=2)

good_matches = [
    m for pair in raw_matches
    if len(pair) == 2
    for m, n in [pair]
    if m.distance < 0.75 * n.distance
]

print("Court-only feature matches:", len(good_matches))

assert len(good_matches) >= 20, "Insufficient court features."

pts_a = np.float32([
    kp_a[m.queryIdx].pt for m in good_matches
])

pts_b = np.float32([
    kp_b[m.trainIdx].pt for m in good_matches
])

# Estimate camera motion
motion_H_court, inlier_mask = cv2.findHomography(
    pts_a,
    pts_b,
    cv2.RANSAC,
    3.0
)

assert motion_H_court is not None

inliers = int(inlier_mask.sum())
ratio = inliers / len(good_matches)

print("RANSAC inliers:", inliers)
print("Inlier ratio:", round(ratio, 4))

# Transfer Frame 100 homography to Frame 101
H_101_court = HOMOGRAPHY @ np.linalg.inv(motion_H_court)

# Visualize inlier matches
inlier_matches = [
    m for m, keep in zip(
        good_matches,
        inlier_mask.ravel()
    )
    if keep
]

matched = cv2.drawMatches(
    frame_a,
    kp_a,
    frame_b,
    kp_b,
    inlier_matches[:80],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(18, 8))
plt.imshow(cv2.cvtColor(matched, cv2.COLOR_BGR2RGB))
plt.title("CourtVision — Court-Only ORB Feature Matching")
plt.axis("off")
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

frame101 = cv2.imread(str(FRAME_PATHS[101]))
assert frame101 is not None

# Original full-frame ORB homography
H_full = HOMOGRAPHY @ np.linalg.inv(motion_H)

# Court-only ORB homography
H_court = HOMOGRAPHY @ np.linalg.inv(motion_H_court)

# Physical free-throw lane corners
lane = COURT_POINTS.astype(np.float32).reshape(-1, 1, 2)

def project_lane(H):
    return cv2.perspectiveTransform(
        lane,
        np.linalg.inv(H)
    ).reshape(-1, 2)

full_points = project_lane(H_full)
court_points = project_lane(H_court)

# Create comparison overlay
overlay = frame101.copy()

cv2.polylines(
    overlay,
    [np.round(full_points).astype(np.int32)],
    True,
    (0, 255, 0),
    3
)

cv2.polylines(
    overlay,
    [np.round(court_points).astype(np.int32)],
    True,
    (0, 0, 255),
    3
)

plt.figure(figsize=(18, 10))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.title(
    "Frame 101 — Green: Full-frame ORB | Red: Court-only ORB"
)
plt.axis("off")

output = "/kaggle/working/courtvision_orb_comparison.png"

plt.savefig(
    output,
    dpi=200,
    bbox_inches="tight"
)

# Save the complete 7-frame court segmentation comparison
output_path = "/kaggle/working/courtvision_court_ORB.png"

plt.tight_layout()
plt.savefig(
    output_path,
    dpi=200,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(f"Image saved to: {output_path}")

print("Saved:", output)

print("\nCorner differences (pixels):")
print(
    np.linalg.norm(
        full_points - court_points,
        axis=1
    ).round(2)
)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Frame 200 landmarks
LINE_POINTS_200 = np.array([
    [302, 648],
    [457, 480],
    [601, 496],
    [1101, 557],
    [471, 675],
    [810, 716],
    [1185, 570],
    [1098, 716]
], dtype=np.float64)

def make_line(a, b):
    return np.cross(
        [a[0], a[1], 1.0],
        [b[0], b[1], 1.0]
    )

def intersect_lines(a, b):
    p = np.cross(a, b)
    if abs(p[2]) < 1e-8:
        raise ValueError("Lines are nearly parallel")
    return p[:2] / p[2]

baseline = make_line(LINE_POINTS_200[0], LINE_POINTS_200[1])
upper = make_line(LINE_POINTS_200[2], LINE_POINTS_200[3])
lower = make_line(LINE_POINTS_200[4], LINE_POINTS_200[5])

IMAGE_POINTS_200 = np.array([
    intersect_lines(baseline, lower),
    intersect_lines(baseline, upper),
    LINE_POINTS_200[6],
    LINE_POINTS_200[7]
], dtype=np.float32)

# Independent candidate homography
H_anchor_200 = cv2.getPerspectiveTransform(
    IMAGE_POINTS_200,
    COURT_POINTS.astype(np.float32)
)

# Previously propagated homography
H_dynamic_200 = homographies[200]

# Project physical lane corners into Frame 200
lane = COURT_POINTS.astype(np.float32).reshape(-1, 1, 2)

dynamic_points = cv2.perspectiveTransform(
    lane,
    np.linalg.inv(H_dynamic_200)
).reshape(-1, 2)

# Draw comparison
frame200 = cv2.imread(str(FRAME_PATHS[200]))
assert frame200 is not None

overlay = frame200.copy()

# Green: independent calibration
cv2.polylines(
    overlay,
    [np.round(IMAGE_POINTS_200).astype(np.int32)],
    True,
    (0, 255, 0),
    3
)

# Red: propagated calibration
cv2.polylines(
    overlay,
    [np.round(dynamic_points).astype(np.int32)],
    True,
    (0, 0, 255),
    3
)

# Quantify disagreement
errors = np.linalg.norm(
    IMAGE_POINTS_200 - dynamic_points,
    axis=1
)

print("Frame 200 calibration comparison")
print("--------------------------------")
print("Corner differences (px):", np.round(errors, 2))
print("Mean difference (px):", round(float(errors.mean()), 2))
print("Maximum difference (px):", round(float(errors.max()), 2))

plt.figure(figsize=(18, 10))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.title(
    "Frame 200 — Green: Independent Anchor | Red: Dynamic Calibration"
)
plt.axis("off")

output_path = "/kaggle/working/courtvision_frame200_drift_comparison.png"

plt.savefig(
    output_path,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

print("Saved:", output_path)

In [ ]:
import cv2
import numpy as np
import json
from pathlib import Path

# Existing calibrated Frame 100
H_anchor_100 = HOMOGRAPHY.copy()

# Independently estimated Frame 200
H_anchor_200 = cv2.getPerspectiveTransform(
    IMAGE_POINTS_200.astype(np.float32),
    COURT_POINTS.astype(np.float32)
)

# Store anchors separately
ANCHORS = {
    100: H_anchor_100,
    200: H_anchor_200
}

# Preserve original homographies
original_homographies = {
    k: v.copy()
    for k, v in homographies.items()
}

print("Calibration anchors registered:")
for idx in sorted(ANCHORS):
    print(f"Frame {idx}: ready")

# Save provisional anchors
anchor_path = Path(
    "/kaggle/working/courtvision_calibration_anchors.json"
)

anchor_path.write_text(
    json.dumps({
        "anchors": {
            str(idx): {
                "homography": H.tolist(),
                "status": (
                    "reference"
                    if idx == 100
                    else "provisional"
                )
            }
            for idx, H in ANCHORS.items()
        }
    }, indent=2)
)

print("Saved:", anchor_path)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

TARGET_FRAME = 150

orb = cv2.ORB_create(nfeatures=5000)
matcher = cv2.BFMatcher(cv2.NORM_HAMMING)

def direct_motion(source_idx, target_idx):
    source = cv2.imread(str(FRAME_PATHS[source_idx]))
    target = cv2.imread(str(FRAME_PATHS[target_idx]))

    gray_source = cv2.cvtColor(source, cv2.COLOR_BGR2GRAY)
    gray_target = cv2.cvtColor(target, cv2.COLOR_BGR2GRAY)

    kp1, des1 = orb.detectAndCompute(gray_source, None)
    kp2, des2 = orb.detectAndCompute(gray_target, None)

    if des1 is None or des2 is None:
        return None, 0, 0

    pairs = matcher.knnMatch(des1, des2, k=2)

    good = [
        m for pair in pairs
        if len(pair) == 2
        for m, n in [pair]
        if m.distance < 0.75 * n.distance
    ]

    if len(good) < 30:
        return None, len(good), 0

    src = np.float32([kp1[m.queryIdx].pt for m in good])
    dst = np.float32([kp2[m.trainIdx].pt for m in good])

    M, mask = cv2.findHomography(
        src, dst, cv2.RANSAC, 3.0
    )

    if M is None or mask is None:
        return None, len(good), 0

    return M, len(good), int(mask.sum())


results = {}

for anchor_idx in [100, 200]:
    M, matches, inliers = direct_motion(
        anchor_idx, TARGET_FRAME
    )

    ratio = inliers / matches if matches else 0

    print(
        f"Anchor {anchor_idx} → Frame {TARGET_FRAME}: "
        f"{matches} matches, {inliers} inliers, "
        f"ratio={ratio:.3f}"
    )

    if M is not None and inliers >= 30 and ratio >= 0.55:
        H_target = ANCHORS[anchor_idx] @ np.linalg.inv(M)
        results[anchor_idx] = H_target


# Visual comparison
frame = cv2.imread(str(FRAME_PATHS[TARGET_FRAME]))
overlay = frame.copy()

colors = {
    100: (0, 255, 0),    # Green
    200: (0, 0, 255)     # Red
}

lane = COURT_POINTS.astype(np.float32).reshape(-1, 1, 2)

projected_by_anchor = {}

for anchor_idx, H in results.items():
    projected = cv2.perspectiveTransform(
        lane,
        np.linalg.inv(H)
    ).reshape(-1, 2)

    if not np.isfinite(projected).all():
        continue

    projected_by_anchor[anchor_idx] = projected

    cv2.polylines(
        overlay,
        [np.round(projected).astype(np.int32)],
        True,
        colors[anchor_idx],
        3
    )

plt.figure(figsize=(18, 10))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.title(
    "Frame 150 — Green: Anchor 100 | Red: Anchor 200"
)
plt.axis("off")

output = "/kaggle/working/courtvision_anchor_comparison_150.png"

plt.savefig(output, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", output)

if 100 in projected_by_anchor and 200 in projected_by_anchor:
    differences = np.linalg.norm(
        projected_by_anchor[100] -
        projected_by_anchor[200],
        axis=1
    )

    print("\nAnchor disagreement at Frame 150:")
    print("Corner differences:", np.round(differences, 2))
    print("Mean difference:", round(float(differences.mean()), 2), "px")
    print("Maximum difference:", round(float(differences.max()), 2), "px")

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Inspect nearby frames
CHECK_FRAMES = list(range(105, 201, 10))

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for ax, idx in zip(axes, CHECK_FRAMES):
    frame = cv2.imread(str(FRAME_PATHS[idx]))

    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {idx}", fontsize=13)
    ax.axis("off")

for ax in axes[len(CHECK_FRAMES):]:
    ax.axis("off")

plt.suptitle(
    "CourtVision — Find a Frame with Visible Lane Landmarks",
    fontsize=17
)

plt.tight_layout()

output = "/kaggle/working/courtvision_anchor_candidates.png"
plt.savefig(output, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", output)

In [ ]:
import cv2
import base64
import re
from pathlib import Path

selector_path = Path("/kaggle/working/courtvision_selector.html")

assert selector_path.exists(), "Original HTML selector not found."

template = selector_path.read_text(encoding="utf-8")

for idx in [105, 115]:
    frame = cv2.imread(str(FRAME_PATHS[idx]))
    assert frame is not None

    image_path = Path(f"/kaggle/working/courtvision_frame{idx}.png")
    cv2.imwrite(str(image_path), frame)

    encoded = base64.b64encode(image_path.read_bytes()).decode("ascii")

    html, count = re.subn(
        r"data:image/(?:png|jpeg|jpg);base64,[A-Za-z0-9+/=]+",
        lambda _: f"data:image/png;base64,{encoded}",
        template
    )

    assert count == 1, f"Could not replace image for Frame {idx}"

    html = html.replace("Frame 100", f"Frame {idx}")

    output = Path(
        f"/kaggle/working/courtvision_selector_frame{idx}.html"
    )

    output.write_text(html, encoding="utf-8")

    print(f"Frame {idx} selector created: {output}")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

LINE_POINTS_105 = np.array([
    [280,471], [323,422],
    [372,370], [622,394],
    [373,482], [543,499],
    [732,518], [793,411]
], dtype=np.float64)

LINE_POINTS_115 = np.array([
    [365,522], [416,466],
    [531,449], [681,466],
    [520,561], [759,588],
    [799,592], [861,486]
], dtype=np.float64)

def line_through(a, b):
    return np.cross(
        [a[0], a[1], 1.0],
        [b[0], b[1], 1.0]
    )

def intersection(a, b):
    p = np.cross(a, b)
    if abs(p[2]) < 1e-8:
        raise ValueError("Lines are nearly parallel")
    return p[:2] / p[2]

def reconstruct_lane(points):
    baseline = line_through(points[0], points[1])
    upper = line_through(points[2], points[3])
    lower = line_through(points[4], points[5])

    return np.array([
        intersection(baseline, lower),
        intersection(baseline, upper),
        points[7],  # Upper free-throw corner
        points[6]   # Lower free-throw corner
    ], dtype=np.float32)

IMAGE_POINTS_105 = reconstruct_lane(LINE_POINTS_105)
IMAGE_POINTS_115 = reconstruct_lane(LINE_POINTS_115)

# The physical lane corners must follow the same order:
# baseline-lower, baseline-upper, FT-upper, FT-lower.
court = np.asarray(COURT_POINTS, dtype=np.float32)
assert court.shape == (4, 2)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, idx, image_points in [
    (axes[0], 105, IMAGE_POINTS_105),
    (axes[1], 115, IMAGE_POINTS_115)
]:
    frame = cv2.imread(str(FRAME_PATHS[idx]))
    assert frame is not None

    overlay = frame.copy()

    # Green = independently selected landmarks
    cv2.polylines(
        overlay,
        [np.round(image_points).astype(np.int32)],
        True,
        (0, 255, 0),
        3
    )

    # Red = existing propagated calibration
    H_dynamic = homographies[idx]

    dynamic_points = cv2.perspectiveTransform(
        court.reshape(-1, 1, 2),
        np.linalg.inv(H_dynamic)
    ).reshape(-1, 2)

    cv2.polylines(
        overlay,
        [np.round(dynamic_points).astype(np.int32)],
        True,
        (0, 0, 255),
        3
    )

    errors = np.linalg.norm(
        image_points - dynamic_points,
        axis=1
    )

    print(f"\nFrame {idx}")
    print("Corner differences:", np.round(errors, 2))
    print("Mean difference:", round(float(errors.mean()), 2), "px")
    print("Maximum difference:", round(float(errors.max()), 2), "px")

    ax.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"Frame {idx} | Mean disagreement: {errors.mean():.1f} px"
    )
    ax.axis("off")

plt.suptitle(
    "CourtVision — Independent Landmarks vs Dynamic Calibration\n"
    "Green: Independent | Red: Propagated",
    fontsize=15
)

plt.tight_layout()

output = "/kaggle/working/courtvision_anchor_validation_105_115.png"

plt.savefig(output, dpi=180, bbox_inches="tight")
plt.show()

print("\nSaved:", output)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

records = []

for idx in range(106, 116):

    M, metrics = estimate_motion(idx - 1, idx)

    if M is None:
        records.append({
            "frame": idx,
            "matches": metrics.get("matches", 0),
            "inliers": metrics.get("inliers", 0),
            "ratio": metrics.get("inlier_ratio", 0),
            "motion_px": np.nan,
            "status": metrics.get("status", "failed")
        })
        continue

    # Measure motion at the reference lane corners
    corners = IMAGE_POINTS.astype(np.float32).reshape(-1, 1, 2)

    moved = cv2.perspectiveTransform(
        corners, M
    ).reshape(-1, 2)

    displacement = np.linalg.norm(
        moved - corners.reshape(-1, 2),
        axis=1
    )

    records.append({
        "frame": idx,
        "matches": metrics.get("matches", 0),
        "inliers": metrics.get("inliers", 0),
        "ratio": metrics.get("inlier_ratio", 0),
        "motion_px": float(displacement.mean()),
        "status": metrics.get("status", "unknown")
    })

import pandas as pd

df = pd.DataFrame(records)

display(df)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(df["frame"], df["ratio"], marker="o")
axes[0].set_title("RANSAC Inlier Ratio — Frames 106–115")
axes[0].set_ylabel("Inlier Ratio")
axes[0].grid(True)

axes[1].plot(df["frame"], df["motion_px"], marker="o")
axes[1].set_title("Estimated Frame-to-Frame Motion")
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Mean Displacement (pixels)")
axes[1].grid(True)

plt.tight_layout()

output = "/kaggle/working/courtvision_drift_diagnostics_105_115.png"
plt.savefig(output, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", output)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Estimate direct motion from Frame 100 to Frame 115
M_direct, matches, inliers = direct_motion(100, 115)

assert M_direct is not None, "Direct matching failed."

ratio = inliers / matches

print("Direct Frame 100 → 115 matching")
print("Matches:", matches)
print("Inliers:", inliers)
print("Inlier ratio:", round(ratio, 4))

# Direct calibration
H_direct_115 = HOMOGRAPHY @ np.linalg.inv(M_direct)

# Existing chained calibration
H_chained_115 = homographies[115]

# Independent manually reconstructed lane
independent_points = IMAGE_POINTS_115.astype(np.float32)

# Court geometry
lane = COURT_POINTS.astype(np.float32).reshape(-1, 1, 2)

def project_lane(H):
    return cv2.perspectiveTransform(
        lane,
        np.linalg.inv(H)
    ).reshape(-1, 2)

direct_points = project_lane(H_direct_115)
chained_points = project_lane(H_chained_115)

# Calculate disagreement with independent landmarks
direct_error = np.linalg.norm(
    direct_points - independent_points,
    axis=1
)

chained_error = np.linalg.norm(
    chained_points - independent_points,
    axis=1
)

print("\nFrame 115 comparison")
print("--------------------------------")
print("Direct matching mean error:", round(float(direct_error.mean()), 2), "px")
print("Chained matching mean error:", round(float(chained_error.mean()), 2), "px")

# Visualize
frame = cv2.imread(str(FRAME_PATHS[115]))
overlay = frame.copy()

def draw_polygon(points, color, thickness=3):
    cv2.polylines(
        overlay,
        [np.round(points).astype(np.int32)],
        True,
        color,
        thickness
    )

draw_polygon(independent_points, (0, 255, 0))  # Green
draw_polygon(chained_points, (0, 0, 255))       # Red
draw_polygon(direct_points, (255, 0, 0))        # Blue

plt.figure(figsize=(18, 10))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))

plt.title(
    "Frame 115 — Green: Independent | "
    "Red: Chained | Blue: Direct"
)

plt.axis("off")

output = "/kaggle/working/courtvision_direct_vs_chained_115.png"

plt.savefig(output, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", output)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

FRAME_INDEX = 115

frame = cv2.imread(str(FRAME_PATHS[FRAME_INDEX]))
assert frame is not None

# Existing manually selected points
LINE_POINTS_115 = np.array([
    [365, 522], [416, 466],  # Baseline
    [531, 449], [681, 466],  # Upper lane boundary
    [520, 561], [759, 588],  # Lower lane boundary
    [799, 592], [861, 486]   # FT corners: lower, upper
], dtype=np.float32)

gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

# Smooth image
blurred = cv2.GaussianBlur(gray, (5, 5), 1.2)

# Detect edges
edges = cv2.Canny(
    blurred,
    threshold1=50,
    threshold2=150
)

# Detect straight line segments
segments = cv2.HoughLinesP(
    edges,
    rho=1,
    theta=np.pi / 180,
    threshold=45,
    minLineLength=35,
    maxLineGap=25
)

assert segments is not None, "No line segments detected."

segments = segments.reshape(-1, 4)

print("Detected line segments:", len(segments))

# Visualize all detected segments
overlay = frame.copy()

for x1, y1, x2, y2 in segments:
    cv2.line(
        overlay,
        (x1, y1),
        (x2, y2),
        (0, 255, 255),
        2
    )

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original Frame 115")
axes[0].axis("off")

axes[1].imshow(edges, cmap="gray")
axes[1].set_title("Canny Edges")
axes[1].axis("off")

axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[2].set_title("Detected Hough Line Segments")
axes[2].axis("off")

plt.tight_layout()

output = "/kaggle/working/courtvision_frame115_hough_lines.png"

plt.savefig(output, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", output)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

REFERENCE_LINES = {
    "baseline": LINE_POINTS_115[0:2],
    "upper_lane": LINE_POINTS_115[2:4],
    "lower_lane": LINE_POINTS_115[4:6],
}

def line_angle(p1, p2):
    return np.arctan2(
        p2[1] - p1[1],
        p2[0] - p1[0]
    )

def angle_difference(a, b):
    return abs(
        (a - b + np.pi / 2) % np.pi - np.pi / 2
    )

def point_line_distance(point, p1, p2):
    direction = p2 - p1
    length = np.linalg.norm(direction)

    if length < 1e-8:
        return np.inf

    return abs(
        np.cross(direction, point - p1)
    ) / length

def detect_reference_line(
    reference,
    segments,
    max_angle_deg=8,
    max_distance=14
):
    p1, p2 = reference.astype(np.float64)

    reference_angle = line_angle(p1, p2)

    accepted = []

    for x1, y1, x2, y2 in segments:
        a = np.array([x1, y1], dtype=np.float64)
        b = np.array([x2, y2], dtype=np.float64)

        segment_length = np.linalg.norm(b - a)

        if segment_length < 25:
            continue

        angle = line_angle(a, b)

        if np.degrees(
            angle_difference(angle, reference_angle)
        ) > max_angle_deg:
            continue

        midpoint = (a + b) / 2

        distance = point_line_distance(
            midpoint, p1, p2
        )

        if distance > max_distance:
            continue

        accepted.extend([a, b])

    if len(accepted) < 4:
        return None, 0

    accepted = np.asarray(accepted, dtype=np.float32)

    # Fit a line to the accepted segment endpoints
    vx, vy, x0, y0 = cv2.fitLine(
        accepted,
        cv2.DIST_HUBER,
        0,
        0.01,
        0.01
    ).flatten()

    direction = np.array([vx, vy], dtype=np.float64)
    origin = np.array([x0, y0], dtype=np.float64)

    return (origin, direction), len(accepted) // 2


detected_lines = {}

for name, reference in REFERENCE_LINES.items():
    fitted, count = detect_reference_line(
        reference,
        segments
    )

    detected_lines[name] = fitted

    print(
        f"{name}: "
        f"{count} matching segments | "
        f"{'DETECTED' if fitted is not None else 'FAILED'}"
    )

# Draw fitted lines
overlay = frame.copy()

colors = {
    "baseline": (0, 255, 0),
    "upper_lane": (255, 0, 0),
    "lower_lane": (0, 0, 255)
}

for name, fitted in detected_lines.items():
    if fitted is None:
        continue

    origin, direction = fitted

    p1 = origin - direction * 1500
    p2 = origin + direction * 1500

    cv2.line(
        overlay,
        tuple(np.round(p1).astype(int)),
        tuple(np.round(p2).astype(int)),
        colors[name],
        2
    )

plt.figure(figsize=(18, 10))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))

plt.title(
    "CourtVision — Automatically Fitted Court Lines\n"
    "Green: Baseline | Blue: Upper Lane | Red: Lower Lane"
)

plt.axis("off")

output = "/kaggle/working/courtvision_frame115_fitted_lines.png"

plt.savefig(output, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", output)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. Convert fitted lines to homogeneous coordinates
# --------------------------------------------------

def homogeneous_line(fitted):
    origin, direction = fitted

    vx, vy = direction
    x0, y0 = origin

    return np.array([
        -vy,
        vx,
        vy * x0 - vx * y0
    ], dtype=np.float64)


def intersect_lines(line1, line2):
    p = np.cross(line1, line2)

    if abs(p[2]) < 1e-8:
        raise ValueError("Lines are nearly parallel")

    return p[:2] / p[2]


required = ["baseline", "upper_lane", "lower_lane"]

for name in required:
    assert detected_lines.get(name) is not None, (
        f"{name} was not detected reliably"
    )

baseline = homogeneous_line(detected_lines["baseline"])
upper = homogeneous_line(detected_lines["upper_lane"])
lower = homogeneous_line(detected_lines["lower_lane"])

# --------------------------------------------------
# 2. Automatically reconstruct baseline intersections
# --------------------------------------------------

baseline_lower = intersect_lines(baseline, lower)
baseline_upper = intersect_lines(baseline, upper)

print("Automatically reconstructed baseline corners:")
print("Lower:", np.round(baseline_lower, 2))
print("Upper:", np.round(baseline_upper, 2))

# --------------------------------------------------
# 3. Compare against manual calibration
# --------------------------------------------------

manual_corners = IMAGE_POINTS_115.astype(np.float64)

automatic_baseline_corners = np.array([
    baseline_lower,
    baseline_upper
])

errors = np.linalg.norm(
    automatic_baseline_corners - manual_corners[:2],
    axis=1
)

print("\nBaseline corner differences:")
print("Lower:", round(float(errors[0]), 2), "px")
print("Upper:", round(float(errors[1]), 2), "px")
print("Mean:", round(float(errors.mean()), 2), "px")

# --------------------------------------------------
# 4. Visualize
# --------------------------------------------------

frame = cv2.imread(str(FRAME_PATHS[115]))
overlay = frame.copy()

# Manual corners: red
for x, y in manual_corners[:2]:
    cv2.circle(
        overlay,
        (int(round(x)), int(round(y))),
        9,
        (0, 0, 255),
        -1
    )

# Automatically reconstructed corners: green
for x, y in automatic_baseline_corners:
    cv2.circle(
        overlay,
        (int(round(x)), int(round(y))),
        7,
        (0, 255, 0),
        -1
    )

plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))

plt.title(
    "Frame 115 — Baseline Corner Validation\n"
    "Green: Automatic | Red: Manual"
)

plt.axis("off")

output = (
    "/kaggle/working/"
    "courtvision_frame115_corner_validation.png"
)

plt.savefig(output, dpi=180, bbox_inches="tight")
plt.show()

print("\nSaved:", output)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Frame 115 manual free-throw corners
# Your original ordering: lower, upper
FT_LOWER = LINE_POINTS_115[6]
FT_UPPER = LINE_POINTS_115[7]

# Reuse our existing reference-guided detector
ft_reference = np.array(
    [FT_UPPER, FT_LOWER],
    dtype=np.float32
)

ft_line, ft_count = detect_reference_line(
    reference=ft_reference,
    segments=segments,
    max_angle_deg=12,
    max_distance=18
)

print("Free-throw line matching segments:", ft_count)

if ft_line is None:
    print("Free-throw line detection FAILED.")
    print("Do not estimate corners from an undetected line.")
else:
    detected_lines["free_throw"] = ft_line

    overlay = frame.copy()

    origin, direction = ft_line

    p1 = origin - direction * 1000
    p2 = origin + direction * 1000

    cv2.line(
        overlay,
        tuple(np.round(p1).astype(int)),
        tuple(np.round(p2).astype(int)),
        (0, 255, 255),
        3
    )

    plt.figure(figsize=(16, 9))
    plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    plt.title("Frame 115 — Detected Free-Throw Line (Yellow)")
    plt.axis("off")

    output = "/kaggle/working/courtvision_frame115_free_throw_line.png"
    plt.savefig(output, dpi=180, bbox_inches="tight")
    plt.show()

    print("Saved:", output)

In [ ]:
# Requires successful detection in Step 1
assert detected_lines.get("free_throw") is not None, (
    "Free-throw line was not detected."
)

ft = homogeneous_line(detected_lines["free_throw"])

ft_upper = intersect_lines(upper, ft)
ft_lower = intersect_lines(lower, ft)

AUTO_CORNERS_115 = np.array([
    baseline_lower,
    baseline_upper,
    ft_upper,
    ft_lower
], dtype=np.float32)

print("Automatically reconstructed lane corners:")
print(np.round(AUTO_CORNERS_115, 2))

manual = IMAGE_POINTS_115.astype(np.float32)

errors = np.linalg.norm(
    AUTO_CORNERS_115 - manual,
    axis=1
)

print("\nCorner differences (px):", np.round(errors, 2))
print("Mean difference:", round(float(errors.mean()), 2), "px")
print("Maximum difference:", round(float(errors.max()), 2), "px")

# Visualize
frame115 = cv2.imread(str(FRAME_PATHS[115]))
overlay = frame115.copy()

cv2.polylines(
    overlay,
    [np.round(AUTO_CORNERS_115).astype(np.int32)],
    True,
    (0, 255, 0),
    3
)

cv2.polylines(
    overlay,
    [np.round(manual).astype(np.int32)],
    True,
    (0, 0, 255),
    2
)

plt.figure(figsize=(17, 9))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.title(
    "Frame 115 — Full Lane Reconstruction\n"
    "Green: Automatic | Red: Manual"
)
plt.axis("off")

output = "/kaggle/working/courtvision_frame115_full_lane.png"
plt.savefig(output, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", output)

In [ ]:
import cv2
import numpy as np

# Reference court coordinates must correspond to:
# baseline-lower, baseline-upper, FT-upper, FT-lower
COURT_LANE = np.asarray(COURT_POINTS, dtype=np.float32)

def refine_frame_calibration(frame_idx, H_initial):
    """
    Refine an existing image-to-court homography
    using court-line detection.

    Returns:
        H_refined, refined_corners, diagnostics

    Returns None for calibration if a required line fails.
    """

    frame = cv2.imread(str(FRAME_PATHS[frame_idx]))
    if frame is None:
        return None, None, {"status": "frame_missing"}

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 1.2)
    edges = cv2.Canny(blurred, 50, 150)

    detected = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=45,
        minLineLength=35,
        maxLineGap=25
    )

    if detected is None:
        return None, None, {"status": "no_segments"}

    segments = detected.reshape(-1, 4)

    # Predict lane corners using existing homography
    predicted = cv2.perspectiveTransform(
        COURT_LANE.reshape(-1, 1, 2),
        np.linalg.inv(H_initial)
    ).reshape(-1, 2)

    # Corner order:
    # 0 baseline-lower
    # 1 baseline-upper
    # 2 FT-upper
    # 3 FT-lower

    reference_lines = {
        "baseline": predicted[[0, 1]],
        "upper_lane": predicted[[1, 2]],
        "lower_lane": predicted[[0, 3]],
        "free_throw": predicted[[2, 3]]
    }

    fitted_lines = {}
    counts = {}

    for name, reference in reference_lines.items():

        fitted, count = detect_reference_line(
            reference=reference,
            segments=segments,
            max_angle_deg=10,
            max_distance=20
        )

        counts[name] = count

        if fitted is None:
            return None, None, {
                "status": f"{name}_failed",
                "counts": counts
            }

        fitted_lines[name] = homogeneous_line(fitted)

    # Reconstruct all four corners
    try:
        corners = np.array([
            intersect_lines(
                fitted_lines["baseline"],
                fitted_lines["lower_lane"]
            ),
            intersect_lines(
                fitted_lines["baseline"],
                fitted_lines["upper_lane"]
            ),
            intersect_lines(
                fitted_lines["upper_lane"],
                fitted_lines["free_throw"]
            ),
            intersect_lines(
                fitted_lines["lower_lane"],
                fitted_lines["free_throw"]
            )
        ], dtype=np.float32)

    except ValueError:
        return None, None, {"status": "intersection_failed"}

    if not np.isfinite(corners).all():
        return None, None, {"status": "invalid_corners"}

    # Prevent unusually large corrections
    displacement = np.linalg.norm(
        corners - predicted,
        axis=1
    )

    mean_correction = float(displacement.mean())
    max_correction = float(displacement.max())

    if max_correction > 100:
        return None, None, {
            "status": "correction_too_large",
            "mean_correction": mean_correction,
            "max_correction": max_correction
        }

    # Compute refined image-to-court homography
    H_refined = cv2.getPerspectiveTransform(
        corners,
        COURT_LANE
    )

    return H_refined, corners, {
        "status": "ok",
        "counts": counts,
        "mean_correction": mean_correction,
        "max_correction": max_correction
    }

In [ ]:
import matplotlib.pyplot as plt

TEST_FRAMES = [105, 110, 115, 120, 125]

fig, axes = plt.subplots(
    1,
    len(TEST_FRAMES),
    figsize=(24, 6)
)

results = {}

for ax, idx in zip(axes, TEST_FRAMES):

    H_initial = homographies[idx]

    H_refined, corners, diagnostics = refine_frame_calibration(
        idx,
        H_initial
    )

    results[idx] = {
        "H": H_refined,
        "corners": corners,
        "diagnostics": diagnostics
    }

    frame = cv2.imread(str(FRAME_PATHS[idx]))
    overlay = frame.copy()

    if corners is not None:
        cv2.polylines(
            overlay,
            [np.round(corners).astype(np.int32)],
            True,
            (0, 255, 0),
            3
        )

        title = (
            f"Frame {idx}\n"
            f"Correction: "
            f"{diagnostics['mean_correction']:.1f}px"
        )
    else:
        title = (
            f"Frame {idx}\n"
            f"{diagnostics['status']}"
        )

    ax.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")

    print(f"Frame {idx}: {diagnostics}")

plt.suptitle(
    "CourtVision — Automatic Court-Line Refinement",
    fontsize=16
)

plt.tight_layout()

output = (
    "/kaggle/working/"
    "courtvision_multiframe_line_refinement.png"
)

plt.savefig(output, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", output)

In [ ]:
import cv2
import base64
from pathlib import Path

FRAME_INDEX = 200

frame = cv2.imread(str(FRAME_PATHS[FRAME_INDEX]))
assert frame is not None, "Frame 200 could not be loaded."

# Save original frame
image_path = Path("/kaggle/working/courtvision_frame_200.png")
cv2.imwrite(str(image_path), frame)

# Load existing standalone selector
selector_path = Path("/kaggle/working/courtvision_selector.html")

assert selector_path.exists(), (
    "Original selector not found. "
    "Run the standalone HTML selector creation cell first."
)

html = selector_path.read_text(encoding="utf-8")

# Replace the embedded image
import re

new_image = base64.b64encode(
    image_path.read_bytes()
).decode("ascii")

html, replacements = re.subn(
    r'data:image/png;base64,[A-Za-z0-9+/=]+',
    lambda _: f"data:image/png;base64,{new_image}",
    html
)

assert replacements == 1, (
    f"Expected one embedded image; found {replacements}."
)

html = html.replace(
    "Frame 100 Landmark Selector",
    "Frame 200 Landmark Selector"
)

output_path = Path(
    "/kaggle/working/courtvision_selector_frame200.html"
)

output_path.write_text(html, encoding="utf-8")

print("Created:", output_path)
print("Image resolution:", frame.shape[1], "x", frame.shape[0])

## 5 — Estimate and validate homography (fails safely if not calibrated)

In [ ]:
CALIBRATED=False; HOMOGRAPHY=None
if IMAGE_POINTS is None or COURT_POINTS is None:
    print('CALIBRATION PENDING: provide 4 verified correspondences in cell 4, then rerun this cell.')
else:
    src=np.asarray(IMAGE_POINTS,dtype=np.float32); dst=np.asarray(COURT_POINTS,dtype=np.float32)
    assert src.shape==(4,2) and dst.shape==(4,2),'Exactly four (x,y) correspondences required'
    assert np.isfinite(src).all() and np.isfinite(dst).all()
    assert np.all((src[:,0]>=0)&(src[:,0]<W)&(src[:,1]>=0)&(src[:,1]<H))
    assert np.all((dst[:,0]>=0)&(dst[:,0]<=COURT_LENGTH_M)&(dst[:,1]>=0)&(dst[:,1]<=COURT_WIDTH_M))
    assert cv2.contourArea(cv2.convexHull(src))>500,'Image landmarks are too close/collinear'
    assert cv2.contourArea(cv2.convexHull(dst))>0.5,'Court landmarks are too close/collinear'
    HOMOGRAPHY=cv2.getPerspectiveTransform(src,dst)
    assert np.isfinite(HOMOGRAPHY).all() and abs(np.linalg.det(HOMOGRAPHY))>1e-10
    reproj=cv2.perspectiveTransform(src[None,:,:],HOMOGRAPHY)[0]
    print('Four-point reprojection errors (m):',np.linalg.norm(reproj-dst,axis=1).round(5))
    print('CAUTION: four-point reprojection error is ~zero by construction. Validate with ADDITIONAL independent landmarks.')
    CALIBRATED=True
    (ROOT/'homography.json').write_text(json.dumps({'H':HOMOGRAPHY.tolist(),'image_points':src.tolist(),'court_points':dst.tolist(),'court_dimensions_m':[COURT_LENGTH_M,COURT_WIDTH_M],'sequence':SEQUENCE_NAME},indent=2))


In [ ]:
# Transform the four selected image points
projected = cv2.perspectiveTransform(
    IMAGE_POINTS.reshape(-1, 1, 2),
    HOMOGRAPHY
).reshape(-1, 2)

print("Projected coordinates:")
print(projected)

print("\nExpected court coordinates:")
print(COURT_POINTS)

error = np.linalg.norm(projected - COURT_POINTS, axis=1)

print("\nMean landmark reprojection error:", error.mean())

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

assert CALIBRATED, "Homography must be calculated first."

# Load calibration frame
frame = cv2.imread(str(FRAME_PATHS[75]))
assert frame is not None

frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

# Create a grid of points inside the calibrated lane
x_values = np.linspace(0, 5.79, 12)
y_values = np.linspace(0, 3.66, 8)

xx, yy = np.meshgrid(x_values, y_values)

court_grid = np.column_stack([
    xx.ravel(),
    yy.ravel()
]).astype(np.float32)

# Transform court coordinates back into image coordinates
H_inv = np.linalg.inv(HOMOGRAPHY)

image_grid = cv2.perspectiveTransform(
    court_grid.reshape(-1, 1, 2),
    H_inv
).reshape(-1, 2)

# Draw the grid over the original image
overlay = frame_rgb.copy()

for x, y in image_grid:
    cv2.circle(
        overlay,
        (int(round(x)), int(round(y))),
        3,
        (0, 255, 0),
        -1
    )

# Plot
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].imshow(overlay)
axes[0].set_title("Frame 75 — Projected Court Grid")
axes[0].axis("off")

axes[1].add_patch(
    Rectangle(
        (0, 0),
        5.79,
        3.66,
        fill=False,
        edgecolor="black",
        linewidth=2
    )
)

axes[1].scatter(
    court_grid[:, 0],
    court_grid[:, 1],
    s=15,
    color="green"
)

axes[1].set_xlim(-0.5, 6.3)
axes[1].set_ylim(4.2, -0.5)
axes[1].set_aspect("equal")
axes[1].set_title("Calibrated Free-Throw Lane — Top View")
axes[1].set_xlabel("Court X (metres)")
axes[1].set_ylabel("Court Y (metres)")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt

candidate_frames = [0, 25, 50, 75, 100, 125, 150, 175, 200, 225, 250, 275, 299]

fig, axes = plt.subplots(4, 4, figsize=(20, 12))

for ax, idx in zip(axes.flat, candidate_frames):
    frame = cv2.imread(str(FRAME_PATHS[idx]))

    if frame is not None:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Frame {idx}")

    ax.axis("off")

for ax in axes.flat[len(candidate_frames):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt

FRAME_INDEX = 100

frame = cv2.imread(str(FRAME_PATHS[FRAME_INDEX]))
assert frame is not None, "Could not load frame"

print(f"Frame dimensions: {frame.shape[1]} × {frame.shape[0]}")

plt.figure(figsize=(18, 10))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title("CourtVision — Frame 100 Calibration")
plt.xlim(0, frame.shape[1])
plt.ylim(frame.shape[0], 0)
plt.xticks(range(0, frame.shape[1] + 1, 100))
plt.yticks(range(0, frame.shape[0] + 1, 50))
plt.grid(alpha=0.25)
plt.show()

In [ ]:
import cv2
from pathlib import Path

frame = cv2.imread(str(FRAME_PATHS[100]))

output_path = "/kaggle/working/courtvision_frame_100.png"

cv2.imwrite(output_path, frame)

print(f"Saved: {output_path}")
print(f"Resolution: {frame.shape[1]} × {frame.shape[0]}")

In [ ]:
from pathlib import Path
import base64

frame_path = Path("/kaggle/working/courtvision_frame_100.png")
assert frame_path.exists(), "Frame 100 image not found"

image_data = base64.b64encode(frame_path.read_bytes()).decode("ascii")

html = """
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>CourtVision Landmark Selector</title>
<style>
body { font-family: Arial; margin: 20px; background: #f5f5f5; }
#imageContainer { position: relative; width: 1280px; max-width: 100%; }
#courtImage { width: 100%; display: block; cursor: crosshair; }
#overlay { position: absolute; inset: 0; width: 100%; height: 100%; pointer-events: none; }
pre { background: #222; color: #90ee90; padding: 15px; }
button { padding: 10px 16px; margin-right: 8px; }
</style>
</head>
<body>

<h2>CourtVision — Frame 100 Landmark Selector</h2>
<p id="instruction"></p>

<div id="imageContainer">
<img id="courtImage" src="data:image/png;base64,IMAGE_DATA">
<canvas id="overlay" width="1280" height="720"></canvas>
</div>

<p>
<button onclick="undo()">Undo</button>
<button onclick="reset()">Reset</button>
<button onclick="copyPoints()">Copy Coordinates</button>
</p>

<pre id="output"></pre>

<script>
const labels = [
  "Baseline point A",
  "Baseline point B",
  "Upper lane point A",
  "Upper lane point B",
  "Lower lane point A",
  "Lower lane point B",
  "Upper free-throw corner",
  "Lower free-throw corner"
];

let points = [];

const img = document.getElementById("courtImage");
const canvas = document.getElementById("overlay");
const ctx = canvas.getContext("2d");

img.addEventListener("click", function(event) {
  if (points.length >= 8) return;

  const rect = img.getBoundingClientRect();
  const x = Math.round((event.clientX - rect.left) * img.naturalWidth / rect.width);
  const y = Math.round((event.clientY - rect.top) * img.naturalHeight / rect.height);

  points.push([x, y]);
  draw();
});

function draw() {
  ctx.clearRect(0, 0, canvas.width, canvas.height);

  points.forEach(([x, y], i) => {
    ctx.beginPath();
    ctx.arc(x, y, 7, 0, Math.PI * 2);
    ctx.fillStyle = "red";
    ctx.fill();

    ctx.font = "bold 20px Arial";
    ctx.fillStyle = "yellow";
    ctx.fillText("P" + (i + 1), x + 12, y - 12);
  });

  const next = points.length < 8
    ? labels[points.length]
    : "Finished! Copy your coordinates.";

  document.getElementById("instruction").textContent =
    "Next click: " + next;

  document.getElementById("output").textContent =
    "LINE_POINTS = " + JSON.stringify(points);
}

function undo() {
  points.pop();
  draw();
}

function reset() {
  points = [];
  draw();
}

function copyPoints() {
  navigator.clipboard.writeText(
    "LINE_POINTS = " + JSON.stringify(points)
  );
}

draw();
</script>
</body>
</html>
"""

html = html.replace("IMAGE_DATA", image_data)

output = Path("/kaggle/working/courtvision_selector.html")
output.write_text(html, encoding="utf-8")

print("Created:", output)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Replace this with the 8 points from your selector.
LINE_POINTS = None

assert LINE_POINTS is not None, "Paste your 8 selected points first."

pts = np.asarray(LINE_POINTS, dtype=np.float64)
assert pts.shape == (8, 2)

def line_through(a, b):
    """Homogeneous line through two image points."""
    a = np.array([a[0], a[1], 1.0])
    b = np.array([b[0], b[1], 1.0])
    return np.cross(a, b)

def intersection(line_a, line_b):
    """Intersection of two nonparallel lines."""
    p = np.cross(line_a, line_b)
    if abs(p[2]) < 1e-8:
        raise ValueError("Selected lines are parallel or nearly parallel.")
    return p[:2] / p[2]

baseline = line_through(pts[0], pts[1])
upper_lane = line_through(pts[2], pts[3])
lower_lane = line_through(pts[4], pts[5])

# Recover obscured lane/baseline intersections.
baseline_upper = intersection(baseline, upper_lane)
baseline_lower = intersection(baseline, lower_lane)

# Free-throw intersections were selected directly.
free_throw_upper = pts[6]
free_throw_lower = pts[7]

IMAGE_POINTS = np.array([
    baseline_lower,     # P1
    baseline_upper,     # P2
    free_throw_upper,   # P3
    free_throw_lower,   # P4
], dtype=np.float32)

print("Estimated lane corners:")
for i, p in enumerate(IMAGE_POINTS, 1):
    print(f"P{i}: ({p[0]:.1f}, {p[1]:.1f})")

# Visual verification
overlay = frame.copy()

cv2.polylines(
    overlay,
    [np.round(IMAGE_POINTS).astype(np.int32)],
    True,
    (0, 255, 0),
    3
)

for i, (x, y) in enumerate(IMAGE_POINTS, 1):
    x, y = int(round(x)), int(round(y))
    cv2.circle(overlay, (x, y), 7, (0, 0, 255), -1)
    cv2.putText(
        overlay, f"P{i}", (x + 10, y - 10),
        cv2.FONT_HERSHEY_SIMPLEX, 0.8,
        (255, 255, 255), 2
    )

plt.figure(figsize=(18, 10))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.title("Frame 100 — Lane Corners Recovered From Court Lines")
plt.axis("off")
plt.show()

## 6 — Optional independent validation landmarks

For meaningful geometric accuracy, enter **at least one additional** visible court landmark not used to fit the homography. Without this, output positions are provisional. An error in court meters does not automatically capture footpoint detection uncertainty.


In [ ]:
VALIDATION_IMAGE_POINTS=None  # [(pixel_x,pixel_y), ...]
VALIDATION_COURT_POINTS=None  # matching court coordinates in meters
if CALIBRATED and VALIDATION_IMAGE_POINTS is not None and VALIDATION_COURT_POINTS is not None:
    vi=np.asarray(VALIDATION_IMAGE_POINTS,np.float32);vc=np.asarray(VALIDATION_COURT_POINTS,np.float32)
    assert vi.ndim==2 and vi.shape[1]==2 and vc.shape==vi.shape
    pred=cv2.perspectiveTransform(vi[None,:,:],HOMOGRAPHY)[0]
    errors=np.linalg.norm(pred-vc,axis=1)
    print('Independent landmark errors (m):',errors.round(3),'| median:',round(float(np.median(errors)),3))
    pd.DataFrame({'error_m':errors}).to_csv(ROOT/'calibration_validation.csv',index=False)
else:print('Independent calibration validation not supplied.')


## 7 — Track players (or reuse Phase 2C tracks)

In [ ]:
if EXISTING_TRACK_CSV:
    tracks=pd.read_csv(EXISTING_TRACK_CSV)
    assert {'frame','track_id','x','y','w','h'}.issubset(tracks.columns)
elif RUN_TRACKING:
    model=YOLO(str(BEST_PT))
    config_src=Path(__import__('ultralytics').__file__).parent/'cfg'/'trackers'/'bytetrack.yaml'
    cfg=yaml.safe_load(config_src.read_text())
    cfg.update({'track_high_thresh':.25,'track_low_thresh':.10,'new_track_thresh':.25,'track_buffer':30,'match_thresh':.90,'fuse_score':True})
    tracker_yaml=ROOT/'bytetrack_optimized.yaml';tracker_yaml.write_text(yaml.safe_dump(cfg,sort_keys=False))
    rows=[]
    for i,path in enumerate(FRAME_PATHS):
        frame=cv2.imread(str(path))
        result=model.track(frame,persist=True,tracker=str(tracker_yaml),imgsz=IMGSZ,conf=CONF,iou=IOU,device=DEVICE,verbose=False)[0]
        if result.boxes is not None and result.boxes.id is not None:
            for tid,box in zip(result.boxes.id.int().cpu().tolist(),result.boxes.xyxy.cpu().numpy()):
                x1,y1,x2,y2=map(float,box)
                rows.append({'frame':int(path.stem),'track_id':int(tid),'x':x1,'y':y1,'w':x2-x1,'h':y2-y1})
        if (i+1)%100==0:print('Tracked',i+1,'/',len(FRAME_PATHS))
    tracks=pd.DataFrame(rows,columns=['frame','track_id','x','y','w','h'])
else:
    raise ValueError('Set RUN_TRACKING=True or provide EXISTING_TRACK_CSV')
tracks.to_csv(ROOT/'tracks_pixel.csv',index=False)
print('Observations:',len(tracks),'Unique track IDs:',tracks.track_id.nunique())


## 8 — Convert player footpoints to court coordinates

In [ ]:
tracks['foot_x_px']=tracks['x']+tracks['w']/2
tracks['foot_y_px']=tracks['y']+tracks['h']
if CALIBRATED:
    pixel=tracks[['foot_x_px','foot_y_px']].to_numpy(np.float32).reshape(1,-1,2)
    mapped=cv2.perspectiveTransform(pixel,HOMOGRAPHY).reshape(-1,2)
    tracks['court_x_m']=mapped[:,0];tracks['court_y_m']=mapped[:,1]
    tracks['inside_court']=(tracks.court_x_m.between(0,COURT_LENGTH_M)&tracks.court_y_m.between(0,COURT_WIDTH_M))
    tracks.to_csv(ROOT/'player_court_coordinates.csv',index=False)
    print('On-court mapped observations:',int(tracks.inside_court.sum()),'/',len(tracks))
else:
    print('No court coordinates exported: calibration is required. Pixel tracks are saved.')


## 9 — Top-down player map and trajectories

In [ ]:
def draw_court(ax):
    from matplotlib.patches import Rectangle, Circle
    ax.add_patch(Rectangle((0,0),COURT_LENGTH_M,COURT_WIDTH_M,fill=False,lw=2))
    ax.plot([COURT_LENGTH_M/2]*2,[0,COURT_WIDTH_M],lw=1.5)
    ax.add_patch(Circle((COURT_LENGTH_M/2,COURT_WIDTH_M/2),1.8,fill=False,lw=1))
    ax.set_xlim(-1,COURT_LENGTH_M+1);ax.set_ylim(COURT_WIDTH_M+1,-1)
    ax.set_aspect('equal');ax.set_xlabel('Court length (m)');ax.set_ylabel('Court width (m)')
if CALIBRATED:
    fig,ax=plt.subplots(figsize=(14,8));draw_court(ax)
    for tid,grp in tracks[tracks.inside_court].groupby('track_id'):
        grp=grp.sort_values('frame')
        if len(grp)<4:continue
        ax.plot(grp.court_x_m,grp.court_y_m,alpha=.5,lw=1)
        ax.scatter(grp.court_x_m.iloc[-1],grp.court_y_m.iloc[-1],s=15)
    ax.set_title('CourtVision — projected footpoint trajectories (not validated player identities)')
    fig.savefig(ROOT/'topdown_trajectories.png',dpi=160,bbox_inches='tight');plt.show()
else:print('Top-down plot skipped until calibrated.')


## 10 — Annotated sample frame and optional court projection overlay

In [ ]:
sample_fid=int(FRAME_PATHS[min(100,len(FRAME_PATHS)-1)].stem)
frame=cv2.imread(str(next(p for p in FRAME_PATHS if int(p.stem)==sample_fid)))
for _,r in tracks[tracks.frame==sample_fid].iterrows():
    x1,y1,x2,y2=int(r.x),int(r.y),int(r.x+r.w),int(r.y+r.h)
    cv2.rectangle(frame,(x1,y1),(x2,y2),(0,220,70),2)
    label=f'ID {int(r.track_id)}'
    if CALIBRATED:label+=f' ({r.court_x_m:.1f},{r.court_y_m:.1f})'
    cv2.putText(frame,label,(x1,max(15,y1-7)),cv2.FONT_HERSHEY_SIMPLEX,.45,(0,220,70),1)
cv2.imwrite(str(ROOT/'player_positions_sample.jpg'),frame)
plt.figure(figsize=(15,8));plt.imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB));plt.axis('off');plt.show()


## 11 — Export and save permanently

In [ ]:
meta={'sequence':SEQUENCE_NAME,'frames':len(FRAME_PATHS),'imgsz':IMGSZ,'conf':CONF,'iou':IOU,
      'calibrated':CALIBRATED,'court_dimensions_m':[COURT_LENGTH_M,COURT_WIDTH_M],
      'limitations':['Court planar homography requires fixed camera or shot-wise recalibration',
      'Four fitted correspondences alone do not validate geometric accuracy',
      'Player footpoint can be inaccurate under occlusion',
      'Track IDs are not guaranteed stable and no metric distance validation has been performed']}
(ROOT/'experiment_config.json').write_text(json.dumps(meta,indent=2))
archive=shutil.make_archive('/kaggle/working/courtvision_phase3_artifacts','zip',root_dir=ROOT.parent,base_dir=ROOT.name)
print('Archive:',archive,'MB:',round(Path(archive).stat().st_size/1e6,2))
print('Kaggle → Save Version with output saving enabled; verify ZIP in saved output.')


## Next decision

1. Review `court_line_diagnostic.jpg` and the first frame to identify four **actual** court landmarks.
2. Enter corresponding pixel and real-court points, rerun cells 5–11, and provide independent validation points.
3. If the camera pans/zooms/cuts, implement per-shot or dynamic calibration **before** deriving movement distances or speeds.
4. Once geometric mapping is validated, proceed to court-aware filtering and a live top-down visualization.
